## Import Libraries

## Define BIDS Layout

In [4]:
import os
import pandas as pd
import numpy as np
import nibabel as nib
from bids import BIDSLayout
from nilearn.glm.first_level import FirstLevelModel

# 1. Setup Layouts and Directories
raw_data_path = '/Volumes/T9/ds001486'
deriv_path = os.path.join(raw_data_path, 'derivatives/fmriprep')

# Initialize BIDS layouts
layout_raw = BIDSLayout(raw_data_path, validate=False)
layout_deriv = BIDSLayout(deriv_path, validate=False, derivatives=False)

bold_files = layout_deriv.get(suffix='bold', extension='nii.gz', desc='preproc', return_type='file')

output_dir = os.path.expanduser('~/Desktop/FINAL_brainmath_GLM_maps')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Define Confounds to Keep 
# Extracting only the essential motion parameters and physiological noise 
# to prevent over-determining the model.
confounds_of_interest = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z', 'csf', 'white_matter']

# 3. Process Each File
for bold_path in bold_files:
    filename = os.path.basename(bold_path)
    
    entities = layout_deriv.parse_file_entities(bold_path)
    sub_id = entities.get('subject')
    ses_id = entities.get('session')
    task_id = entities.get('task')
    run_id = entities.get('run') # PyBIDS handles the integer/string formatting automatically

    # Task Filtering (Only Math tasks)
    if task_id not in ['Mult', 'Sub']:
        continue

    print(f"Processing sub-{sub_id}, Task: {task_id}, Run: {run_id}...")

    # 1. Locate Events (Raw Layout)
    events_files = layout_raw.get(subject=sub_id, session=ses_id, task=task_id, run=run_id, suffix='events', extension='tsv', return_type='file')
    events_path = events_files[0] if events_files else None

    # 2. Locate Confounds (Derivatives Layout - Notice we don't look for 'space')
    confound_files = layout_deriv.get(subject=sub_id, session=ses_id, task=task_id, run=run_id, desc='confounds', extension='tsv', return_type='file')
    confound_path = confound_files[0] if confound_files else None

    # 3. Locate Mask (Derivatives Layout - We DO look for matching 'space' and 'res')
    mask_files = layout_deriv.get(subject=sub_id, session=ses_id, task=task_id, run=run_id, space=entities.get('space'), res=entities.get('res'), desc='brain', suffix='mask', extension='nii.gz', return_type='file')
    mask_path = mask_files[0] if mask_files else None

    # 4. Check for missing files with detailed errors
    if not events_path:
        print(f"  -> Skipping: Missing EVENTS file.")
        continue
    if not confound_path:
        print(f"  -> Skipping: Missing CONFOUNDS file.")
        continue
    if not mask_path:
        print(f"  -> Skipping: Missing MASK file.")
        continue

    try:
        # Load Data
        events_df = pd.read_csv(events_path, sep='\t')
        confounds_df = pd.read_csv(confound_path, sep='\t')
        
        # Filter confounds to only the essentials that actually exist in the file
        available_confounds = [c for c in confounds_of_interest if c in confounds_df.columns]
        selected_confounds = confounds_df[available_confounds].fillna(0)

        # Extract TR directly from the NIfTI header (the most reliable method)
        img = nib.load(bold_path)
        t_r = img.header.get_zooms()[3]

        # 4. GLM Estimation
        fmri_glm = FirstLevelModel(
            t_r=t_r,
            mask_img=mask_path,       # Focuses the GLM only on brain tissue
            noise_model='ar1',        # Handles temporal autocorrelation
            standardize=True,         # Comparable scale across subjects
            smoothing_fwhm=5          # 5mm smoothing 
        )

        # Fit the GLM
        fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)

        # 5. Compute Contrast
        # Nilearn looks for the condition name inside the 'trial_type' column of events.tsv.
        # This dynamically grabs the first active condition name if task_id isn't an exact match.
        active_conditions = events_df['trial_type'].unique()
        target_condition = task_id if task_id in active_conditions else active_conditions[0]

        # Extract the Z-score map for the target task
        beta_map = fmri_glm.compute_contrast(target_condition, output_type='z_score')

        # 6. Save the Output
        unique_id = f"sub-{sub_id}_ses-{ses_id}_task-{task_id}_run-{run_id}"
        save_path = os.path.join(output_dir, f'{unique_id}_GLM_zmap.nii.gz')
        nib.save(beta_map, save_path)
        print(f"  -> Saved GLM Map: {save_path}")

    except Exception as e:
        print(f"  -> Error on sub-{sub_id}: {e}")

Processing sub-007, Task: Mult, Run: 01...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: The following unexpected columns in events data will be ignored: prime_stim, response_time, accuracy, target_stim
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: Duplicated events were detected. Amplitudes of these events will be summed. You might want to verify your inputs.
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)


  -> Saved GLM Map: /Users/jchong058/Desktop/FINAL_brainmath_GLM_maps/sub-007_ses-T1_task-Mult_run-01_GLM_zmap.nii.gz
Processing sub-007, Task: Mult, Run: 02...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: The following unexpected columns in events data will be ignored: prime_stim, response_time, accuracy, target_stim
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: Duplicated events were detected. Amplitudes of these events will be summed. You might want to verify your inputs.
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)


  -> Saved GLM Map: /Users/jchong058/Desktop/FINAL_brainmath_GLM_maps/sub-007_ses-T1_task-Mult_run-02_GLM_zmap.nii.gz
Processing sub-007, Task: Sub, Run: 01...


...

  -> Saved GLM Map: /Users/jchong058/Desktop/FINAL_brainmath_GLM_maps/sub-106_ses-T2_task-Mult_run-02_GLM_zmap.nii.gz
Processing sub-106, Task: Sub, Run: 01...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: The following unexpected columns in events data will be ignored: prime_stim, response_time, accuracy, target_stim
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: Duplicated events were detected. Amplitudes of these events will be summed. You might want to verify your inputs.
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)


  -> Saved GLM Map: /Users/jchong058/Desktop/FINAL_brainmath_GLM_maps/sub-106_ses-T2_task-Sub_run-01_GLM_zmap.nii.gz
Processing sub-106, Task: Sub, Run: 02...


/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: The following unexpected columns in events data will be ignored: prime_stim, response_time, accuracy, target_stim
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)
/var/folders/nb/t9_yz5pn7ql2nyvm9cy5kngc0000gn/T/ipykernel_99932/3075317524.py:89: UserWarning: Duplicated events were detected. Amplitudes of these events will be summed. You might want to verify your inputs.
  fmri_glm = fmri_glm.fit(bold_path, events=events_df, confounds=selected_confounds)


  -> Saved GLM Map: /Users/jchong058/Desktop/FINAL_brainmath_GLM_maps/sub-106_ses-T2_task-Sub_run-02_GLM_zmap.nii.gz
